# PSM Memory LOCOMO benchmark on Colab

This notebook installs the published npm packages, downloads the PSM GGUF model through `psm-memory setup`, ingests LOCOMO turns into a SQLite memory DB, and evaluates evidence retrieval.

Recommended Colab runtime: GPU. Start with a small `LIMIT` smoke test before running the full LOCOMO set.

In [1]:
# Install Node.js 22. Colab's default Node can be older than the package engine range.
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

2026-05-14 16:19:48 - Installing pre-requisites
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https:/

In [2]:
# Keep model and embedding caches under /content so the paths are predictable.
%env PSM_MEMORY_HOME=/content/psm-memory-cache
!mkdir -p /content/psm-memory-work /content/locomo/results
%cd /content/psm-memory-work
!npm init -y
!npm install @psm-memory/cli@latest @psm-memory/sdk@latest

env: PSM_MEMORY_HOME=/content/psm-memory-cache
/content/psm-memory-work
⠙Wrote to /content/psm-memory-work/package.json:

{
  "name": "psm-memory-work",
  "version": "1.0.0",
  "description": "",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC"
}



⠙⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏npm warn deprecated prebuild-install@7.1.3: No longer maintained. Please contact the author of the relevant native addon; alternatives are available.
⠏⠋⠙⠹npm warn deprecated boolean@3.2.0: Package no longer supported. Contact Support at https://www.npmjs.com/support for more info.
⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴

In [3]:
# Ensure the published CLI can prepare the default PSM model and embedding model.
!npx psm-memory setup --pretty

⠙PSM model already installed: /content/psm-memory-cache/psm-memory-qwen-1.5b-q4_k_m.gguf
Preparing embedding model: Xenova/all-MiniLM-L6-v2
{
  "model": "/content/psm-memory-cache/psm-memory-qwen-1.5b-q4_k_m.gguf",
  "embedding_model": "Xenova/all-MiniLM-L6-v2",
  "installed": true
}
⠙

In [4]:
# Clone only for benchmark data and the Colab harness. The benchmark imports @psm-memory/sdk from npm.
!rm -rf /content/PSM
!git clone --depth 1 https://github.com/chkrishna2001/PSM.git /content/PSM
!cp /content/PSM/benchmark/locomo/colab/locomo-benchmark.mjs /content/psm-memory-work/locomo-benchmark.mjs
!ls -lh /content/PSM/benchmark/locomo/data/locomo10.json /content/psm-memory-work/locomo-benchmark.mjs

Cloning into '/content/PSM'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 151 (delta 15), reused 107 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 980.37 KiB | 3.97 MiB/s, done.
Resolving deltas: 100% (15/15), done.
-rw-r--r-- 1 root root 2.7M May 14 16:22 /content/PSM/benchmark/locomo/data/locomo10.json
-rw-r--r-- 1 root root 8.4K May 14 16:22 /content/psm-memory-work/locomo-benchmark.mjs


In [7]:
# Smoke ingest. Increase LIMIT after this succeeds. Use LIMIT=0 for the full dataset.
LIMIT = 0
DB = "/content/locomo/results/locomo-psm-memory.db"
MODEL = "/content/psm-memory-cache/psm-memory-qwen-1.5b-q4_k_m.gguf"
DATA = "/content/PSM/benchmark/locomo/data/locomo10.json"

!node /content/psm-memory-work/locomo-benchmark.mjs ingest --data "$DATA" --db "$DB" --model "$MODEL" --limit $LIMIT --batch-size 5

node-llama-cpp supported GPU backends: cuda, false
node-llama-cpp selected backend: cuda
[warn] load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
node-llama-cpp model GPU layers: 29
ingested=5 stored=5 ignored=0 failed=0
ingested=10 stored=10 ignored=0 failed=0
ingested=15 stored=15 ignored=0 failed=0
ingested=20 stored=20 ignored=0 failed=0
ingested=25 stored=25 ignored=0 failed=0
ingested=30 stored=30 ignored=0 failed=0
ingested=35 stored=35 ignored=0 failed=0
ingested=40 stored=40 ignored=0 failed=0
ingested=45 stored=45 ignored=0 failed=0
ingested=50 stored=50 ignored=0 failed=0
ingested=55 stored=55 ignored=0 failed=0
ingested=60 stored=60 ignored=0 failed=0
ingested=65 stored=65 ignored=0 failed=0
ingested=70 stored=70 ignored=0 failed=0
ingested=75 stored=75 ignored=0 failed=0
ingested=80 stored=80 ignored=0 failed=0
ingested=85 stored=85 ignored=0 failed=0
ingested=90 stored=90 ignored=0 failed=0
in

In [6]:
!node /content/psm-memory-work/locomo-benchmark.mjs evaluate --data "$DATA" --db "$DB" --out /content/locomo/results/locomo-results.json --top-k 3
!npx psm-memory show --db "$DB" --table episodic --limit 10 --pretty

{
  "questions": 197,
  "hit_at_1": 0.025380710659898477,
  "hit_at_3": 0.03553299492385787
}
Wrote /content/locomo/results/locomo-results.json
⠙[
  {
    "id": "f2a0fc90-770f-4953-aa6d-39c86fc4abca",
    "user_id": "locomo-conv-26",
    "content": "Melanie: Thanks, Caroline. It's still a work in progress, but I'm doing my best. My kids are so excited about summer break! We're thinking about going camping next month. Any fun plans for the summer?",
    "strength": 0.75,
    "decay_rate": 0.02,
    "emotional_weight": 0.1,
    "confidence": 0.5,
    "tags": "[\"parse_fallback\",\"locomo_sample_id:conv-26\",\"locomo_dia_id:D2:7\",\"locomo_speaker:Melanie\"]",
    "created_at": "2026-05-14 16:24:58",
    "last_accessed": null,
    "promoted": 0
  },
  {
    "id": "75c2677d-e44f-4de8-becf-c2b619630502",
    "user_id": "locomo-conv-26",
    "content": "Caroline: That's great, Mel! Taking time for yourself is so important. You're doing an awesome job looking after yourself and your family!",

In [ ]:
# Evaluate evidence retrieval over the memory DB.
OUT = "/content/locomo/results/locomo-results.json"
!node /content/psm-memory-work/locomo-benchmark.mjs evaluate --data "$DATA" --db "$DB" --out "$OUT" --top-k 3

In [ ]:
# Inspect output artifacts.
!ls -lh /content/locomo/results
!python - <<'PY'
import json
from pathlib import Path
summary = Path('/content/locomo/results/ingest-summary.json')
results = Path('/content/locomo/results/locomo-results.json')
if summary.exists():
    print('ingest summary')
    print(json.dumps(json.loads(summary.read_text()), indent=2)[:4000])
if results.exists():
    print('evaluation summary')
    print(json.dumps(json.loads(results.read_text())['summary'], indent=2))
PY

For a full run, set `LIMIT = 0` in the ingest cell and rerun ingest + evaluate. The full run can take hours depending on Colab GPU availability and `node-llama-cpp` backend selection.